# FENRIX Multi-Company Anonymization (Reproducible)

**Professor/team-friendly Colab notebook.**

Reproduces the masked/anonymized submission artifact with stated limitations.

**Outputs:** `anonymized_bundle.zip` containing sanitized SEC summaries, news briefs, and binned metrics for 8 companies.

**Important limitations:**
- This is a **masked/anonymized submission artifact with stated limitations**.
- No mathematical anonymity guarantee is claimed.
- NVIDIA QA is optional; if `NVIDIA_API_KEY` is not set, QA status is `INCOMPLETE`.
- Public output excludes originals, private maps, raw filings, API keys, and local paths.

## 0. Setup

Install the repository and dependencies. Set your SEC user agent before running.

In [ ]:
import os
import sys
from pathlib import Path

# Install the package from GitHub (or local checkout).
REPO_URL = os.environ.get(
    "FENRIX_REPO_URL", "https://github.com/Scott-Switzer/fenrix-synthetic-data.git"
)
REPO_BRANCH = os.environ.get("FENRIX_REPO_BRANCH", "feature/colab-multicompany-anonymization")

REPO_DIR = Path("/content/fenrix-synthetic-data")
if not REPO_DIR.exists():
    !git clone --branch {REPO_BRANCH} {REPO_URL} {REPO_DIR}
else:
    print(f"Repo already present at {REPO_DIR}")

%cd {REPO_DIR}
!pip install -q -e ".[dev,submission]"
sys.path.insert(0, str(REPO_DIR / "src"))

## 1. Configure Credentials

**Required:** `SEC_USER_AGENT` — a contact email per SEC fair-access policy.

**Optional:** `NVIDIA_API_KEY` — if set, enables NVIDIA artifact verification. If absent, the build still succeeds with QA status `INCOMPLETE`.

In [ ]:
# REQUIRED: set your SEC user agent (email contact).
os.environ["SEC_USER_AGENT"] = os.environ.get("SEC_USER_AGENT", "Your Name your-email@example.com")

# OPTIONAL: set NVIDIA_API_KEY as a Colab secret or env var.
# Leave unset to run without NVIDIA QA (status will be INCOMPLETE).
nvidia_key_set = bool(os.environ.get("NVIDIA_API_KEY", "").strip())
print(f"SEC_USER_AGENT: set ({len(os.environ['SEC_USER_AGENT'])} chars)")
print(f"NVIDIA_API_KEY: {'set' if nvidia_key_set else 'not set (QA will be INCOMPLETE)'}")

## 2. Build the Anonymized Artifact

Runs `build_submission_fast.py` for 8 tickers over 10 years of data.

In [ ]:
TICKERS = "CL,PEP,TJX,PM,AMZN,HBAN,BLK,GOOGL"
OUTPUT_ROOT = Path("/content/fenrix_output")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

!python scripts/build_submission_fast.py \
    --tickers {TICKERS} \
    --output-root {OUTPUT_ROOT} \
    --years 10 \
    --news-limit 5 \
    --enable-nvidia-qa auto

## 3. Run Local Validation Scans

These scans confirm no SEC cover-page fields, raw filing headers, XBRL garbage, API keys, or local paths leak into public output.

In [ ]:
import re

SEC_COVER_PATTERN = re.compile(
    r"IRS Employer|Employer Identification|Commission File|Exact name of registrant|"
    r"principal executive offices|Address of principal executive offices|telephone number|"
    r"FORM 8-K|FORM 10-Q|FORM 10-K|SIGNATURES|/s/|Registrant|Hudson Yards|Mountain View|"
    r"Framingham|Cochituate|New York, New York|CA 94043|253-0000|390-1000|"
    r"([0-9]{3})[[:space:]]*[0-9]{3}-[0-9]{4}|[0-9]{2}-[0-9]{7}|"
    r"[0-9]{1,5}[[:space:]][A-Za-z0-9 .-]+(Road|Street|Avenue|Ave|Blvd|Drive|Lane|Way|Yards)"
)
BROAD_PATTERN = re.compile(
    r"us-gaap:|xbrli:|iso4217:|utr:|srt:|Exact name of registrant|Commission File Number|"
    r"IRS Employer|Employer Identification|PricewaterhouseCoopers|Deloitte|Ernst|KPMG|"
    r"/s/|SIGNATURES|FORM 8-K|FORM 10-Q|FORM 10-K|http|www\.|nvapi-|NVIDIA_API_KEY|"
    r"/Users/|/content/"
)


def scan_dir(root: Path, pattern: re.Pattern[str], label: str) -> int:
    hits = 0
    for path in sorted(root.rglob("*")):
        if not path.is_file() or path.suffix.lower() not in {".md", ".json", ".csv", ".txt"}:
            continue
        text = path.read_text(encoding="utf-8", errors="replace")
        for match in pattern.finditer(text):
            print(f"  [{label}] {path.relative_to(OUTPUT_ROOT)}: {match.group(0)!r}")
            hits += 1
    return hits


sec_hits = scan_dir(OUTPUT_ROOT / "anonymized", SEC_COVER_PATTERN, "SEC_COVER")
broad_hits = scan_dir(OUTPUT_ROOT, BROAD_PATTERN, "BROAD")
print(f"\nSEC cover-page scan hits: {sec_hits}")
print(f"Broad artifact scan hits: {broad_hits}")
assert sec_hits == 0, "SEC cover-page leakage detected!"
print("Validation scans passed.")

## 4. Preview One Company

Display the sanitized SEC summaries for COMPANY_001.

In [ ]:
company_dir = OUTPUT_ROOT / "anonymized" / "COMPANY_001" / "sec"
for md_file in sorted(company_dir.glob("*.md")):
    print(f"\n{'=' * 60}\n{md_file.name}\n{'=' * 60}")
    print(md_file.read_text(encoding="utf-8")[:800])

## 5. Download the Anonymized Bundle

Download `anonymized_bundle.zip` to your local machine.

In [ ]:
from google.colab import files

ZIP_PATH = OUTPUT_ROOT / "exports" / "anonymized_bundle.zip"
assert ZIP_PATH.exists(), f"ZIP not found at {ZIP_PATH}"
print(f"ZIP size: {ZIP_PATH.stat().st_size:,} bytes")
files.download(str(ZIP_PATH))

## Limitations

- This artifact is a **masked/anonymized submission artifact with stated limitations**.
- No mathematical anonymity guarantee is claimed or implied.
- NVIDIA QA is optional. Without `NVIDIA_API_KEY`, QA status is `INCOMPLETE`.
- Public output excludes originals, private maps, raw filings, API keys, and local paths.
- The artifact is intended for research review, not production release.